In [ ]:
# One Colab cell (T4 GPU). Orchestrator replay on Sri's saved 6-round trajectories (150 questions).
# Step A (GPU, ~5 min): score every unique draft with Strands + bge top-2 (question+draft), our best free critic.
# Step B (CPU, seconds): replay stopping/routing policies over the saved rounds and compare F1 vs. compute.
# Replay is valid for stopping: stopping only cuts a saved trajectory short; it never changes earlier rounds.
# No API keys, no paid models. Saves scores to Drive so Step B can be re-run without the GPU.

import subprocess, sys, os, json, glob, time
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "strands-decider", "datasets"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/haltiq_sri_trajectory_strands.csv"

if not os.path.exists("/content/haltiq-sri"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", "sri",
                    "https://github.com/acm-research-f26/Halt-IQ", "/content/haltiq-sri"], check=True)
TRAJ = [json.load(open(f)) for f in sorted(glob.glob("/content/haltiq-sri/implementation-3/study150/cache/hotpot_*.json"))]
print("trajectories:", len(TRAJ))

import pandas as pd, numpy as np, requests, torch

# ---------- Step A: Strands scores for unique (question, draft) pairs ----------
pairs = sorted({(t["id"], r["answer"]) for t in TRAJ for r in t["rounds"]})
have = pd.read_csv(OUT) if os.path.exists(OUT) else pd.DataFrame(columns=["id", "answer", "strands"])
have_keys = set(zip(have.id, have.answer.astype(str)))
todo = [p for p in pairs if p not in have_keys]
print("unique drafts:", len(pairs), "| still to score:", len(todo))

if todo:
    assert torch.cuda.is_available(), "Set Runtime > Change runtime type > T4 GPU, then rerun."
    from datasets import load_dataset
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    log = open("/content/server.log", "w")
    srv = subprocess.Popen(["strands-decider", "serve", "StrandsAgents/strands-decider-2B-hobson-v19",
                            "--port", "8000"], stdout=log, stderr=log)
    for i in range(60):
        time.sleep(5)
        if srv.poll() is not None: raise SystemExit(open("/content/server.log").read()[-1500:])
        try: requests.get("http://localhost:8000", timeout=2); print("server up after", (i + 1) * 5, "s"); break
        except requests.exceptions.ConnectionError: pass

    def S(question, evidence, draft):
        state = f"Question: {question}\nEvidence: {evidence}\nDraft answer: {draft}"
        a = requests.post("http://localhost:8000/v1/systemone", json={"state": state, "questions": {"sufficient": {
            "type": "noul", "instructions": "Is the draft answer correct and fully supported by the evidence?"}}}
        ).json()["answers"]["sufficient"]
        return a.get("noul", a.get("value"))

    need = {t["id"].removeprefix("hotpot_") for t in TRAJ}
    hq = load_dataset("hotpotqa/hotpot_qa", "distractor", split="validation").filter(lambda i: i in need, input_columns="id")
    paras = {"hotpot_" + ex["id"]: [f"{t}: {' '.join(s)}" for t, s in zip(ex["context"]["title"], ex["context"]["sentences"])] for ex in hq}
    qtext = {t["id"]: t["question"] for t in TRAJ}
    print("questions with passages:", sum(t["id"] in paras for t in TRAJ), "/", len(TRAJ))
    bn = "BAAI/bge-reranker-base"
    btok = AutoTokenizer.from_pretrained(bn)
    bmod = AutoModelForSequenceClassification.from_pretrained(bn).cuda().eval()

    @torch.no_grad()
    def top2(qid, draft):
        ps = paras[qid]
        x = btok([f"{qtext[qid]} {draft}"] * len(ps), ps, truncation=True, max_length=512, padding=True, return_tensors="pt").to("cuda")
        s = bmod(**x).logits.view(-1).float().tolist()
        return "\n".join(ps[i] for i in sorted(range(len(ps)), key=lambda i: -s[i])[:2])

    rows = have.to_dict("records")
    for n, (qid, ans) in enumerate(todo, 1):
        rows.append({"id": qid, "answer": ans, "strands": S(qtext[qid], top2(qid, ans), ans)})
        if n % 25 == 0 or n == len(todo):
            pd.DataFrame(rows).to_csv(OUT, index=False); print(f"{n}/{len(todo)} scored", flush=True)
    srv.kill()

score = {(r.id, str(r.answer)): float(r.strands) for r in pd.read_csv(OUT).itertuples()}

# ---------- Step B: replay policies (no model calls) ----------
T_HI = 0.8   # team's declared threshold, fixed in advance (not tuned on these trajectories)

def wtok(r): return r["writer"]["usage"]["total_tokens"]
def ctok(r): return r["critic"]["usage"]["total_tokens"]

def replay(t, policy):
    """Returns (final F1, final EM, LLM tokens, Strands calls, rounds used)."""
    R = t["rounds"]; tokens = 0; calls = 0
    for k, r in enumerate(R):
        tokens += wtok(r)
        last = k == len(R) - 1
        if policy == "fixed1": stop = True
        elif policy == "fixed6": tokens += ctok(r); stop = last
        elif policy == "sri_llm_critic":                       # stop when the LLM critic approves
            tokens += ctok(r); stop = r["critic_approved"] or last
        elif policy == "answer_unchanged":                     # cheap rule: stop once a revision repeats the answer
            stop = last or (k > 0 and r["answer"].strip().lower() == R[k - 1]["answer"].strip().lower())
            if not stop: tokens += ctok(r)                     # reviewer feedback needed for the next revision
        elif policy in ("strands_gate", "orchestrator"):
            calls += 1; s = score[(t["id"], str(r["answer"]))]
            if policy == "strands_gate": stop = s >= T_HI or last
            else:                                              # confident -> stop now; otherwise revise until the answer stops changing
                same = k > 0 and r["answer"].strip().lower() == R[k - 1]["answer"].strip().lower()
                stop = last or s >= T_HI or same
            if not stop: tokens += ctok(r)
        if stop:
            return r["quality"], r["answer_em"], tokens, calls, k + 1

POL = ["fixed1", "fixed6", "sri_llm_critic", "answer_unchanged", "strands_gate", "orchestrator"]
rng = np.random.default_rng(0)
for split in ["test", "all"]:
    S_ = [t for t in TRAJ if split == "all" or t["split"] == split]
    res = {p: np.array([replay(t, p) for t in S_], float) for p in POL}
    f6 = res["fixed6"][:, 2].mean()
    au = res["answer_unchanged"]
    print(f"\n=== {split} ({len(S_)} questions) — oracle F1 {np.mean([max(r['quality'] for r in t['rounds']) for t in S_]):.3f}")
    print(f"{'policy':18s} {'F1':>6s} {'EM':>6s} {'LLM tokens':>11s} {'% of fixed6':>11s} {'Strands calls':>14s} {'rounds':>7s}  F1 vs fixed6 [95% CI]")
    for p in POL:
        a = res[p]; d = a[:, 0] - res["fixed6"][:, 0]
        b = [d[rng.integers(0, len(d), len(d))].mean() for _ in range(2000)]
        print(f"{p:18s} {a[:,0].mean():6.3f} {a[:,1].mean():6.3f} {a[:,2].mean():11.0f} {a[:,2].mean()/f6:10.0%} "
              f"{a[:,3].mean():14.2f} {a[:,4].mean():7.2f}  {d.mean():+.3f} [{np.percentile(b,2.5):+.3f}, {np.percentile(b,97.5):+.3f}]")
    o = res["orchestrator"]
    print(f"orchestrator vs answer_unchanged: F1 {o[:,0].mean()-au[:,0].mean():+.3f}, LLM tokens {o[:,2].mean()/au[:,2].mean()-1:+.0%}, "
          f"questions stopped at round 1: {(o[:,4]==1).sum()} vs {(au[:,4]==1).sum()}, "
          f"round-1 stops that were wrong (F1<0.5) but a later round was right: "
          f"{sum(1 for t,row in zip(S_,o) if row[4]==1 and t['rounds'][0]['quality']<0.5 and max(r['quality'] for r in t['rounds'])>=0.5)}")

Mounted at /content/drive
trajectories: 150
unique drafts: 166 | still to score: 166
server up after 65 s


README.md:   0%|          | 0.00/9.52k [00:00<?, ?B/s]

distractor/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

distractor/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

distractor/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  166MB            

...

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

25/166 scored
50/166 scored
75/166 scored
100/166 scored
125/166 scored
150/166 scored
166/166 scored

=== test (80 questions) — oracle F1 0.392
policy                 F1     EM  LLM tokens % of fixed6  Strands calls  rounds  F1 vs fixed6 [95% CI]
fixed1              0.359  0.212        1514         8%           0.00    1.00  -0.033 [-0.069, -0.004]
fixed6              0.392  0.237       18227       100%           0.00    6.00  +0.000 [+0.000, +0.000]
sri_llm_critic      0.359  0.212        3320        18%           0.00    1.12  -0.033 [-0.071, -0.004]
answer_unchanged    0.392  0.237        5076        28%           0.00    2.16  +0.000 [+0.000, +0.000]
strands_gate        0.378  0.225       10295        56%           4.00    4.00  -0.014 [-0.037, +0.000]
orchestrator        0.378  0.225        3515        19%           1.70    1.70  -0.014 [-0.038, +0.000]
orchestrator vs answer_unchanged: F1 -0.014, LLM tokens -31%, questions stopped at round 1: 32 vs 0, round-1 stops that were wro